In [ ]:
# ==========================================
# Data Consolidation - Merge Multiple CSV Files
# ==========================================
import pandas as pd
import glob
import os
import re
from google.colab import drive

#Mount Google Drive
drive.mount('/content/drive')

#Set folder path (ensure this matches your actual Drive structure)
folder_path = "/content/drive/MyDrive/EEG_Project/"

#Use glob to match all chunk files
file_pattern = os.path.join(folder_path, "onEEGWaveLAD_Metrics_Sub_*_*.csv")
csv_files = glob.glob(file_pattern)

#Custom sort function: extract the first number after 'Sub_'
def get_start_subject_id(filepath):
    filename = os.path.basename(filepath)
    #Extract first digit after 'Sub_' (e.g., 'Sub_1_to_3.csv' -> 1)
    match = re.search(r'Sub_(\d+)_', filename)
    if match:
        return int(match.group(1))
    return 0

#Sort files by starting subject number (ascending)
csv_files.sort(key=get_start_subject_id)

print(f"Found {len(csv_files)} chunk files to merge, sorted by subject order:")
for f in csv_files:
    print(f"  - {os.path.basename(f)}")

if len(csv_files) == 0:
    print("[ERROR] No matching files found. Please check folder_path.")
else:
    #Read all CSVs and concatenate in correct order
    df_list = []
    for file in csv_files:
        temp_df = pd.read_csv(file)
        df_list.append(temp_df)

    #Concatenate row-wise
    master_df = pd.concat(df_list, ignore_index=True)
    print(f"\nOriginal merged data shape: {master_df.shape}")

    #Filter out EOG channels
    if 'Channel' in master_df.columns:
        df_clean = master_df[~master_df['Channel'].str.contains('EOG', case=False, na=False)]
        print(f"Cleaned data shape (EOG channels removed): {df_clean.shape}")
    else:
        df_clean = master_df
        print("[WARNING] 'Channel' column not found. Skipping EOG filtering.")

    #Save consolidated file for 30 subjects
    clean_filename = os.path.join(folder_path, "onEEGWaveLAD_Metrics_MASTER_30_Subjects.csv")
    df_clean.to_csv(clean_filename, index=False)
    print(f"\n[SUCCESS] Consolidated dataset saved to:\n  {clean_filename}")

In [ ]:
# ==========================================
# Stage 2 Refined Grid - Combine Bs = 4, 6, 8, 10, 12, 14, 16, 18
# ==========================================
import pandas as pd
import glob
import os
import re
from google.colab import drive

drive.mount('/content/drive')

#Set paths
folder_path = "/content/drive/MyDrive/EEG_Project/"
stage1_csv = os.path.join(folder_path, "onEEGWaveLAD_Metrics_AVGREF_MASTER_30_Subjects.csv")
output_csv = os.path.join(folder_path, "onEEGWaveLAD_Metrics_STAGE2_REFINED.csv")

#Extract Bs = 4, 8, 16 from Stage 1 master file
print(f"Reading Stage 1 master data...")
if os.path.exists(stage1_csv):
    df_stage1 = pd.read_csv(stage1_csv)
    target_stage1_bs = [4, 8, 16]
    df_extracted = df_stage1[df_stage1['Bs'].isin(target_stage1_bs)].copy()
    print(f"  [SUCCESS] Extracted Bs={target_stage1_bs}, {len(df_extracted)} rows.")
else:
    print(f"[ERROR] File not found: {stage1_csv}")
    df_extracted = pd.DataFrame()

#Read and merge new chunk files (filter for Bs = 6, 10, 12, 14, 18)
file_pattern = os.path.join(folder_path, "onEEGWaveLAD_Metrics_POST_REF_Sub_*_*.csv")
new_csv_files = glob.glob(file_pattern)

def get_start_subject_id(filepath):
    filename = os.path.basename(filepath)
    match = re.search(r'Sub_(\d+)_', filename)
    if match:
        return int(match.group(1))
    return 0

new_csv_files.sort(key=get_start_subject_id)
print(f"\nFound {len(new_csv_files)} chunk files, scanning for refined grid data:")

df_new_list = []
for f in new_csv_files:
    print(f"  - Reading {os.path.basename(f)}")
    temp_df = pd.read_csv(f)
    df_new_list.append(temp_df)

if len(df_new_list) > 0:
    df_new = pd.concat(df_new_list, ignore_index=True)

    #Remove EOG channels (if present)
    if 'Channel' in df_new.columns:
        df_new_clean = df_new[~df_new['Channel'].str.contains('EOG', case=False, na=False)]
    else:
        df_new_clean = df_new

    #Filter: keep only Bs = 6, 10, 12, 14, 18
    target_new_bs = [6, 10, 12, 14, 18]
    df_new_clean = df_new_clean[df_new_clean['Bs'].isin(target_new_bs)].copy()
    print(f"  [SUCCESS] Extracted and cleaned Bs={target_new_bs}, {len(df_new_clean)} rows.")

    #Combine Stage 1 (4, 8, 16) + new grid (6, 10, 12, 14, 18)
    df_final = pd.concat([df_extracted, df_new_clean], ignore_index=True)

    #Sort by Subject (numeric), Channel (alphabetic), Bs (ascending)
    print("\nSorting merged data for clean structure...")
    df_final['Sub_Num'] = df_final['Subject'].str.extract(r'(\d+)').astype(int)
    df_final = df_final.sort_values(by=['Sub_Num', 'Channel', 'Bs']).drop(columns=['Sub_Num'])

    #Save final output
    df_final.to_csv(output_csv, index=False)
    print(f"\n[SUCCESS] Final dataset with Bs=[4, 6, 8, 10, 12, 14, 16, 18] saved to:\n  {output_csv}")
    print("\nYou can now run convergence/lower bound analysis scripts with this STAGE2 file.")
else:
    print("[ERROR] No chunk files found. Please check folder path.")

In [ ]:
# ==========================================
# Extreme Value Diagnostics - Multi-level Threshold Analysis
# ==========================================
import os
import pandas as pd
import numpy as np

#Mount Google Drive
try:
    from google.colab import drive
    drive.mount('/content/drive')
    print("[SUCCESS] Google Drive mounted.\n")
except ImportError:
    pass

ROOT_PATH = "/content/drive/MyDrive/EEG_Project/"
csv_file = os.path.join(ROOT_PATH, "onEEGWaveLAD_Metrics_STAGE2_REFINED.csv")

if os.path.exists(csv_file):
    df = pd.read_csv(csv_file)
    target_metrics = ['SNR-Diff', 'PSNR-Diff']

    #Define multi-level thresholds for outlier detection
    thresholds = [80, 50, 30, 10]

    print("-" * 65)
    print("Extreme Value Distribution Diagnostic Report")
    print("-" * 65)

    for m in target_metrics:
        if m in df.columns:
            v_min = df[m].min()
            v_max = df[m].max()
            total_rows = len(df)

            print(f"\nMetric: [{m}]")
            print(f"  Range: [{v_min:.3f}, {v_max:.3f}]")

            #Calculate key percentiles to understand distribution
            p = df[m].quantile([0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
            print(f"  Percentile Distribution:")
            print(f"      1st percentile  : {p[0.01]:>7.3f}  (1% of data is worse than this)")
            print(f"      5th percentile  : {p[0.05]:>7.3f}")
            print(f"     25th percentile  : {p[0.25]:>7.3f}")
            print(f"     50th percentile  : {p[0.50]:>7.3f}  (median)")
            print(f"     75th percentile  : {p[0.75]:>7.3f}")
            print(f"     95th percentile  : {p[0.95]:>7.3f}")
            print(f"     99th percentile  : {p[0.99]:>7.3f}  (1% of data is better than this)")

            #Multi-level threshold check (count values exceeding each threshold)
            print(f"  Threshold Analysis:")
            for t in thresholds:
                n_pos = (df[m] >= t).sum()
                n_neg = (df[m] <= -t).sum()
                n_total = n_pos + n_neg
                pct = (n_total / total_rows) * 100
                print(f"      |value| >= {t:>2} : {n_total:>4} rows ({pct:>5.2f}%)  "
                      f"[positive>+{t}: {n_pos:<3}, negative<-{t}: {n_neg:<3}]")

            print("-" * 65)
else:
    print(f"[ERROR] File not found: {csv_file}")